# Gap diagnostic validation

This notebook conservatively revalidates every detected gap. The revised classes describe collection behavior and resource evidence only; they are **not** slowdown targets.

### 1. Define paths and fingerprint every protected input

**What the code does:** Imports the required libraries, defines repository-relative paths, and calculates SHA-256 checksums before analysis.  
**Why it is needed:** The validation must be reproducible and must not alter either metrics dataset.  
**How to interpret the output:** The fingerprints identify the exact gap report and datasets used for validation.

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
GAP_INPUT_PATH = PROJECT_ROOT / "reports" / "gap_investigation.csv"
SEGMENTED_INPUT_PATH = PROJECT_ROOT / "data" / "interim" / "segmented_metrics.csv"
CLEANED_PROTECTED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_metrics.csv"
VALIDATION_OUTPUT_PATH = PROJECT_ROOT / "reports" / "gap_validation.csv"
SUMMARY_OUTPUT_PATH = PROJECT_ROOT / "reports" / "gap_validation_summary.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

protected_paths = [GAP_INPUT_PATH, SEGMENTED_INPUT_PATH, CLEANED_PROTECTED_PATH]
missing_paths = [str(path) for path in protected_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Required inputs are missing: {missing_paths}")

hashes_before = {path: sha256_file(path) for path in protected_paths}
for path, fingerprint in hashes_before.items():
    print(f"{path}: {fingerprint}")

/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/gap_investigation.csv: 5845f5928b90ac708c6727fbd60aba856202446f891cf5346c5e508df68f890e
/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/interim/segmented_metrics.csv: eafbd4f01ddf3f1b7bd5d0df11ce4b4e467c679e7718539ddbdf39b4ce6a8578
/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/processed/cleaned_metrics.csv: 801976ddbe91a7415ce84038ff7df5b89c601e2201e56eaf5503154f93c2e577


### 2. Load copies and validate the required evidence

**What the code does:** Reads the gap investigation and segmented metrics, then checks that timestamps, window metrics, diagnostic fields, and sequence keys exist.  
**Why it is needed:** Every revised decision must be based on traceable evidence rather than assumptions.  
**How to interpret the output:** The shapes show that one validation decision will be produced for every input gap while the metrics remain read-only.

In [2]:
gap_input = pd.read_csv(GAP_INPUT_PATH)
segmented_input = pd.read_csv(SEGMENTED_INPUT_PATH)

required_gap_columns = {
    "machine_id", "run_id", "timestamp_before_gap", "timestamp_after_gap",
    "gap_seconds", "gap_threshold_seconds", "window_30s_row_count",
    "cpu_pct_30s_mean", "ram_pct_30s_mean", "swap_pct_30s_mean",
    "disk_latency_ms_30s_mean", "disk_latency_ms_30s_max",
    "context_switches_per_s_30s_mean", "missed_deadline_30s_count",
    "missed_deadline_30s_rate", "diagnostic_class"
}
required_metric_columns = {"machine_id", "run_id", "timestamp", "sensor_errors_json", "context_switches_per_s"}
missing_gap_columns = required_gap_columns - set(gap_input.columns)
missing_metric_columns = required_metric_columns - set(segmented_input.columns)
if missing_gap_columns or missing_metric_columns:
    raise KeyError({
        "missing_gap_columns": sorted(missing_gap_columns),
        "missing_metric_columns": sorted(missing_metric_columns),
    })

gap_validation = gap_input.copy(deep=True)
metrics_analysis = segmented_input.copy(deep=True)
gap_validation["gap_id"] = np.arange(1, len(gap_validation) + 1)

print(f"Detected gaps to validate: {len(gap_validation)}")
print(f"Segmented metric rows available as evidence: {len(metrics_analysis):,}")

Detected gaps to validate: 357
Segmented metric rows available as evidence: 31,059


### 3. Build run-specific context-switch baselines

**What the code does:** Safely parses timestamps and calculates each run's median and 99th-percentile context-switch rate.  
**Why it is needed:** Context-switch levels differ greatly across machines, so an absolute universal threshold would be misleading.  
**How to interpret the output:** Context switching is considered strongly abnormal only when the pre-gap mean reaches the run's 99th percentile and is at least 1.5× its median.

In [3]:
metrics_analysis["_timestamp_dt"] = pd.to_datetime(metrics_analysis["timestamp"], errors="coerce", utc=True)
gap_validation["_before_dt"] = pd.to_datetime(gap_validation["timestamp_before_gap"], errors="coerce", utc=True)
gap_validation["_after_dt"] = pd.to_datetime(gap_validation["timestamp_after_gap"], errors="coerce", utc=True)

group_keys = ["machine_id", "run_id"]
context_baselines = (
    metrics_analysis.groupby(group_keys)["context_switches_per_s"]
    .agg(
        context_switch_run_median="median",
        context_switch_run_p99=lambda values: values.quantile(0.99),
    )
    .reset_index()
)
gap_validation = gap_validation.merge(context_baselines, on=group_keys, how="left", validate="many_to_one")

print("Run-specific context-switch baselines:")
display(context_baselines)

Run-specific context-switch baselines:


,machine_id,run_id,context_switch_run_median,context_switch_run_p99
0,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,504.30,3970.413
1,0890dcc046c079acc4de4202,89cdc34b-e02b-43b4-9284-144276df508a,432.00,5270.438
2,7232bc533c21ce408d45d473,19127a70-e60c-4b47-b3e6-71e89175c174,9778.45,15752.348
3,7232bc533c21ce408d45d473,d88b15dd-1915-43ca-90ef-69f31ff4d9c1,7029.40,33643.416
4,7232bc533c21ce408d45d473,ec61755d-b5be-42ac-875d-3123e92add7a,3875.50,19317.940
5,7232bc533c21ce408d45d473,fac82c2e-545a-402a-80f8-3d1fccb72c68,8348.30,29612.100
6,a0f8c86097e55fbfa506d057,2e9f4457-2a7f-4647-87ed-b86ac6343d33,40658.90,119844.850
7,a0f8c86097e55fbfa506d057,373070d1-ba59-4244-88ab-2d44a21f4983,2668.10,12678.420
8,a0f8c86097e55fbfa506d057,70577f8e-1430-4489-8602-2096521ab84e,37910.00,115111.496
9,a0f8c86097e55fbfa506d057,f24e9c1a-f11e-405c-b663-45fa6e25c405,19756.20,112551.280


### 4. Recover exact sensor-timeout evidence from each 30-second window

**What the code does:** Reopens the same pre-gap windows in the segmented metrics and distinguishes general sensor errors, any timeout, and temperature-specific timeout rows.  
**Why it is needed:** Timeouts are collection-quality evidence and must not be counted as independent resource pressure.  
**How to interpret the output:** The three count columns show whether a gap is plausibly dominated by blocked sensor collection.

In [4]:
def inspect_sensor_error(value):
    try:
        parsed = json.loads(value) if pd.notna(value) else {}
    except (TypeError, json.JSONDecodeError):
        return True, False, False
    if not isinstance(parsed, dict) or not parsed:
        return bool(parsed), False, False
    any_timeout = False
    temperature_timeout = False
    for key, message in parsed.items():
        message_lower = str(message).lower()
        is_timeout = "timeout" in message_lower or "timed out" in message_lower
        any_timeout = any_timeout or is_timeout
        temperature_timeout = temperature_timeout or (is_timeout and "temp" in str(key).lower())
    return True, any_timeout, temperature_timeout

metrics_by_run = {
    key: group.sort_values("_timestamp_dt")
    for key, group in metrics_analysis.groupby(group_keys, sort=False)
}
sensor_window_records = []
for _, row in gap_validation.iterrows():
    run_metrics = metrics_by_run.get((row["machine_id"], row["run_id"]), metrics_analysis.iloc[0:0])
    window_start = row["_before_dt"] - pd.Timedelta(seconds=30)
    window = run_metrics.loc[run_metrics["_timestamp_dt"].between(window_start, row["_before_dt"], inclusive="both")]
    inspected = window["sensor_errors_json"].apply(inspect_sensor_error)
    sensor_window_records.append({
        "sensor_error_rows_30s_validated": int(sum(result[0] for result in inspected)),
        "sensor_timeout_rows_30s": int(sum(result[1] for result in inspected)),
        "temperature_timeout_rows_30s": int(sum(result[2] for result in inspected)),
    })

gap_validation = pd.concat(
    [gap_validation.reset_index(drop=True), pd.DataFrame(sensor_window_records)], axis=1
)
print(f"Gaps with any sensor timeout in the preceding window: {gap_validation['sensor_timeout_rows_30s'].gt(0).sum()}")
print(f"Gaps with a temperature timeout: {gap_validation['temperature_timeout_rows_30s'].gt(0).sum()}")

Gaps with any sensor timeout in the preceding window: 295
Gaps with a temperature timeout: 283


### 5. Calculate independent resource-pressure signals

**What the code does:** Creates separate Boolean signals for sustained CPU, RAM, meaningful swap, abnormal disk latency, and strongly abnormal context switching. Missed deadlines and sensor timeouts are stored separately.  
**Why it is needed:** Strong pressure must be supported by different resource groups; collection lateness cannot be used to manufacture slowdown evidence.  
**How to interpret the output:** `resource_pressure_groups` names every triggered group. CPU+RAM alone remains weak; strong pressure needs at least two groups and at least one of swap, disk, or context switching.

In [5]:
# Conservative, explicit thresholds applied to the preceding 30-second means.
gap_validation["cpu_pressure_signal"] = gap_validation["cpu_pct_30s_mean"].ge(80)
gap_validation["ram_pressure_signal"] = gap_validation["ram_pct_30s_mean"].ge(90)
gap_validation["swap_pressure_signal"] = gap_validation["swap_pct_30s_mean"].ge(50)
gap_validation["disk_pressure_signal"] = (
    gap_validation["disk_latency_ms_30s_mean"].ge(10)
    | gap_validation["disk_latency_ms_30s_max"].ge(100)
)
gap_validation["context_switch_pressure_signal"] = (
    gap_validation["context_switches_per_s_30s_mean"].ge(gap_validation["context_switch_run_p99"])
    & gap_validation["context_switches_per_s_30s_mean"].ge(1.5 * gap_validation["context_switch_run_median"])
)

resource_signal_columns = {
    "cpu": "cpu_pressure_signal",
    "ram": "ram_pressure_signal",
    "swap": "swap_pressure_signal",
    "disk": "disk_pressure_signal",
    "context_switch": "context_switch_pressure_signal",
}
gap_validation["resource_pressure_signal_count"] = gap_validation[list(resource_signal_columns.values())].sum(axis=1).astype(int)
gap_validation["resource_pressure_groups"] = gap_validation.apply(
    lambda row: ";".join(name for name, column in resource_signal_columns.items() if row[column]), axis=1
)
gap_validation["missed_deadline_timing_signal"] = (
    gap_validation["missed_deadline_30s_count"].ge(2)
    | gap_validation["missed_deadline_30s_rate"].ge(0.25)
)
gap_validation["sensor_timeout_collection_signal"] = gap_validation["sensor_timeout_rows_30s"].gt(0)
gap_validation["strong_independent_resource_pressure"] = (
    gap_validation["resource_pressure_signal_count"].ge(2)
    & gap_validation[["swap_pressure_signal", "disk_pressure_signal", "context_switch_pressure_signal"]].any(axis=1)
)
gap_validation["weak_resource_pressure_signal"] = (
    gap_validation["resource_pressure_signal_count"].ge(1)
    & ~gap_validation["strong_independent_resource_pressure"]
)

signal_counts = gap_validation[list(resource_signal_columns.values())].sum().rename("gap_count")
display(signal_counts.to_frame())

,gap_count
cpu_pressure_signal,308
ram_pressure_signal,35
swap_pressure_signal,0
disk_pressure_signal,0
context_switch_pressure_signal,0


### 6. Apply the revised diagnostic hierarchy

**What the code does:** Reclassifies each gap using evidence sufficiency, duration, independent resource signals, and timeout evidence in a documented order.  
**Why it is needed:** Ordering resolves conflicts conservatively: multi-minute gaps without pressure are pause/sleep; truly independent pressure outranks timeout; timeout outranks weak pressure for likely collection delays.  
**How to interpret the output:** `revised_classification_reason` states the exact condition used for every row; none of these classes is a final slowdown label.

In [6]:
main_window_columns = [
    "cpu_pct_30s_mean", "ram_pct_30s_mean", "swap_pct_30s_mean",
    "disk_latency_ms_30s_mean", "context_switches_per_s_30s_mean"
]
gap_validation["evidence_sufficient"] = (
    gap_validation["window_30s_row_count"].ge(2)
    & gap_validation[main_window_columns].notna().any(axis=1)
)
gap_validation["short_gap_ceiling_seconds"] = np.maximum(
    2.0 * gap_validation["gap_threshold_seconds"], 30.0
)

def revised_decision(row):
    groups = row["resource_pressure_groups"] or "none"
    if not row["evidence_sufficient"]:
        return "insufficient_evidence", "Fewer than two pre-gap rows or all principal resource-window values are missing."
    if row["gap_seconds"] >= 300 and row["resource_pressure_signal_count"] == 0:
        return "likely_pause_or_sleep", "Gap is at least five minutes with no independent resource-pressure signal."
    if row["strong_independent_resource_pressure"]:
        return "strong_resource_pressure", f"At least two independent resource groups triggered, including swap/disk/context: {groups}."
    if row["sensor_timeout_collection_signal"]:
        return "collection_delay_sensor_timeout", f"Pre-gap sensor timeout evidence is present; resource groups ({groups}) do not meet the strong rule."
    if row["weak_resource_pressure_signal"]:
        return "weak_resource_pressure", f"Resource evidence is limited to {groups}; CPU alone or CPU+RAM alone is not strong evidence."
    if row["gap_seconds"] <= row["short_gap_ceiling_seconds"]:
        return "small_sampling_delay", "Short gap is near the run threshold with no independent resource pressure or sensor timeout."
    return "insufficient_evidence", "Gap is not short and has no sufficient evidence for another diagnostic class."

decisions = gap_validation.apply(revised_decision, axis=1, result_type="expand")
gap_validation[["revised_diagnostic_class", "revised_classification_reason"]] = decisions
gap_validation = gap_validation.rename(columns={"diagnostic_class": "old_diagnostic_class"})
gap_validation["diagnostic_only_not_target"] = True

allowed_classes = [
    "collection_delay_sensor_timeout", "small_sampling_delay", "likely_pause_or_sleep",
    "weak_resource_pressure", "strong_resource_pressure", "insufficient_evidence"
]
assert gap_validation["revised_diagnostic_class"].isin(allowed_classes).all()
assert len(gap_validation) == len(gap_input)

print("Revised class counts:")
display(gap_validation["revised_diagnostic_class"].value_counts().reindex(allowed_classes, fill_value=0).to_frame("gap_count"))

Revised class counts:


,gap_count
revised_diagnostic_class,
collection_delay_sensor_timeout,294
small_sampling_delay,23
likely_pause_or_sleep,3
weak_resource_pressure,37
strong_resource_pressure,0
insufficient_evidence,0


### 7. Compare old and revised decisions and save both reports

**What the code does:** Builds a complete transition table, includes zero-count revised classes, saves row-level validation evidence, and validates both CSVs after reading them back.  
**Why it is needed:** The comparison makes the correction auditable and prevents silent loss of gaps or decision categories.  
**How to interpret the output:** Transition counts show exactly how earlier `possible_slowdown` proposals changed under the stricter independent-evidence rules.

In [7]:
revised_counts = gap_validation["revised_diagnostic_class"].value_counts().reindex(allowed_classes, fill_value=0)
class_count_summary = pd.DataFrame({
    "summary_type": "revised_class_count",
    "old_classification": "",
    "revised_classification": revised_counts.index,
    "gap_count": revised_counts.values,
})
transition_summary = (
    gap_validation.groupby(["old_diagnostic_class", "revised_diagnostic_class"])
    .size().rename("gap_count").reset_index()
    .rename(columns={
        "old_diagnostic_class": "old_classification",
        "revised_diagnostic_class": "revised_classification",
    })
)
transition_summary.insert(0, "summary_type", "old_to_revised")
gap_validation_summary = pd.concat([class_count_summary, transition_summary], ignore_index=True)
gap_validation_summary["percentage_of_all_gaps"] = (
    gap_validation_summary["gap_count"] / len(gap_validation) * 100
).round(4)

output_columns_to_drop = ["_before_dt", "_after_dt"]
gap_validation_output = gap_validation.drop(columns=output_columns_to_drop)
gap_validation_output.to_csv(VALIDATION_OUTPUT_PATH, index=False)
gap_validation_summary.to_csv(SUMMARY_OUTPUT_PATH, index=False)

saved_validation = pd.read_csv(VALIDATION_OUTPUT_PATH)
saved_summary = pd.read_csv(SUMMARY_OUTPUT_PATH)
assert len(saved_validation) == len(gap_input)
assert saved_validation["gap_id"].is_unique
assert set(saved_validation["revised_diagnostic_class"]) <= set(allowed_classes)
assert len(saved_summary.loc[saved_summary["summary_type"].eq("revised_class_count")]) == len(allowed_classes)

print("Old-to-revised comparison:")
display(pd.crosstab(gap_validation["old_diagnostic_class"], gap_validation["revised_diagnostic_class"]))
print(f"Saved: {VALIDATION_OUTPUT_PATH}")
print(f"Saved: {SUMMARY_OUTPUT_PATH}")

Old-to-revised comparison:


revised_diagnostic_class,collection_delay_sensor_timeout,likely_pause_or_sleep,small_sampling_delay,weak_resource_pressure
old_diagnostic_class,,,,
likely_pause_or_sleep,0,1,0,0
possible_slowdown,285,2,6,37
small_sampling_delay,9,0,17,0


Saved: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/gap_validation.csv
Saved: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/gap_validation_summary.csv


### 8. Display the requested validation summary and prove non-modification

**What the code does:** Prints every revised class count, key evidence totals, and final protected-file checksum comparison.  
**Why it is needed:** This provides a clear human-review handoff and confirms that validation created reports only.  
**How to interpret the output:** Timeout, weak-pressure, and strong-pressure totals are diagnostic counts; the final checksum line must be `True`.

In [8]:
timeout_class_count = int(gap_validation["revised_diagnostic_class"].eq("collection_delay_sensor_timeout").sum())
weak_pressure_count = int(gap_validation["revised_diagnostic_class"].eq("weak_resource_pressure").sum())
strong_pressure_count = int(gap_validation["revised_diagnostic_class"].eq("strong_resource_pressure").sum())
hashes_after = {path: sha256_file(path) for path in protected_paths}
protected_inputs_unchanged = hashes_before == hashes_after

print("FINAL GAP VALIDATION SUMMARY")
for class_name, count in revised_counts.items():
    print(f"{class_name}: {int(count)}")
print(f"Gaps classified mainly as sensor-timeout collection delay: {timeout_class_count}")
print(f"Gaps with weak resource pressure: {weak_pressure_count}")
print(f"Gaps with strong independent resource pressure: {strong_pressure_count}")
print(f"Protected datasets and input report were not modified: {protected_inputs_unchanged}")
print("These revised classes are diagnostics only, not a final slowdown target.")

assert protected_inputs_unchanged, "A protected input file changed during validation."
display(pd.crosstab(gap_validation["old_diagnostic_class"], gap_validation["revised_diagnostic_class"]))

FINAL GAP VALIDATION SUMMARY
collection_delay_sensor_timeout: 294
small_sampling_delay: 23
likely_pause_or_sleep: 3
weak_resource_pressure: 37
strong_resource_pressure: 0
insufficient_evidence: 0
Gaps classified mainly as sensor-timeout collection delay: 294
Gaps with weak resource pressure: 37
Gaps with strong independent resource pressure: 0
Protected datasets and input report were not modified: True
These revised classes are diagnostics only, not a final slowdown target.


revised_diagnostic_class,collection_delay_sensor_timeout,likely_pause_or_sleep,small_sampling_delay,weak_resource_pressure
old_diagnostic_class,,,,
likely_pause_or_sleep,0,1,0,0
possible_slowdown,285,2,6,37
small_sampling_delay,9,0,17,0
